# Prueba de integración con Google Colab

Comprueba el acceso al repositorio, la instalación del paquete `slm`, una operación CUDA real y la persistencia del resultado en Google Drive. Ejecuta este notebook con un runtime de Google Colab o con un kernel remoto de Colab desde VS Code.

In [16]:
import platform
import sys

import torch

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
cuda_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_available else None
print(f"CUDA disponible: {cuda_available}")
print(f"GPU: {gpu_name or 'ninguna'}")

checks = {
    "GPU disponible": cuda_available,
    "Repositorio accesible": False,
    "Paquete slm importable": False,
    "Código de src/slm ejecutado": False,
    "Operación CUDA ejecutada": False,
    "Google Drive montado": False,
    "Resultado escrito en Drive": False,
    "Resultado leído de Drive": False,
}
if not cuda_available:
    print("FAIL: se requiere una GPU CUDA para esta prueba.")

Python: 3.12.13
PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: Tesla T4


In [17]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/miguel-conde/SLM.git"
REPO_NAME = "SLM"
REPO_DIR = Path("/content") / REPO_NAME

if REPO_DIR.exists():
    subprocess.run(
        ["git", "pull", "--ff-only"],
        cwd=REPO_DIR,
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(REPO_DIR)
git_commit = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
checks["Repositorio accesible"] = bool(git_commit)
print(f"Repositorio: {REPO_DIR}")
print(f"Commit: {git_commit}")

Repositorio: /content/SLM
Commit: 581f231


In [18]:
%pip install --no-deps .

Processing /content/SLM
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for slm: filename=slm-0.1.0-py3-none-any.whl size=7354 sha256=62a833ba08703fc865b6d47465fdbbd8964184a785d6023f38de78e69b747ab6
  Stored in directory: /tmp/pip-ephem-wheel-cache-vgig8ido/wheels/f8/5a/8c/06d20e167d06935fe4e1c484f012de077894c57d126e43fc1d
Successfully built slm
  Attempting uninstall: slm
    Found existing installation: slm 0.1.0
    Uninstalling slm-0.1.0:
      Successfully uninstalled slm-0.1.0


In [19]:
from slm.integration_test import gpu_matrix_test

checks["Paquete slm importable"] = True
print("PASS: slm.integration_test se importó desde el paquete instalado.")

PASS: slm.integration_test se importó desde el paquete instalado.


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"PASS: dispositivo seleccionado: {device}")
else:
    print("FAIL: sólo hay CPU; selecciona un runtime Colab con GPU.")

PASS: dispositivo seleccionado: cuda


In [21]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/SLM")
RESULTS_DIR = DRIVE_ROOT / "results" / "integration_test"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
checks["Google Drive montado"] = Path("/content/drive/MyDrive").is_dir()
print(f"Directorio de resultados: {RESULTS_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directorio de resultados: /content/drive/MyDrive/SLM/results/integration_test


In [22]:
result = None
if device.type == "cuda":
    try:
        result = gpu_matrix_test(device)
    except RuntimeError as error:
        result = {"status": "FAIL", "error": str(error)}
        print(f"FAIL: la operación CUDA no se completó: {error}")
    else:
        print(result)
        checks["Código de src/slm ejecutado"] = result.get("status") == "PASS"
        checks["Operación CUDA ejecutada"] = result.get("device") == "cuda"
else:
    result = {"status": "FAIL", "error": "No hay GPU CUDA disponible."}
    print("FAIL: se omite la multiplicación porque no hay GPU CUDA.")

{'device': 'cuda', 'shape': [2, 2], 'sum': 134.0, 'mean': 33.5, 'status': 'PASS'}


In [23]:
from datetime import datetime, timezone
import json

timestamp = datetime.now(timezone.utc).isoformat()
filename_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
result_path = RESULTS_DIR / f"colab_integration_{filename_timestamp}.json"
payload = {
    "timestamp": timestamp,
    "git_commit": git_commit,
    "python_version": platform.python_version(),
    "pytorch_version": str(torch.__version__),
    "gpu": gpu_name,
    "result": result,
    "final_status": "PENDING",
}

try:
    with result_path.open("w", encoding="utf-8") as result_file:
        json.dump(payload, result_file, indent=2, ensure_ascii=False)
    checks["Resultado escrito en Drive"] = result_path.is_file()
    with result_path.open(encoding="utf-8") as result_file:
        loaded_payload = json.load(result_file)
    checks["Resultado leído de Drive"] = (
        loaded_payload.get("git_commit") == git_commit
        and loaded_payload.get("result") == result
    )
except (OSError, json.JSONDecodeError, TypeError) as error:
    print(f"FAIL: no se pudo guardar o leer el JSON: {error}")

payload["final_status"] = "PASS" if all(checks.values()) else "FAIL"
try:
    with result_path.open("w", encoding="utf-8") as result_file:
        json.dump(payload, result_file, indent=2, ensure_ascii=False)
    with result_path.open(encoding="utf-8") as result_file:
        final_payload = json.load(result_file)
    checks["Resultado leído de Drive"] = (
        checks["Resultado leído de Drive"]
        and final_payload.get("final_status") == payload["final_status"]
        and final_payload.get("result") == result
    )
except (OSError, json.JSONDecodeError, TypeError) as error:
    checks["Resultado escrito en Drive"] = False
    checks["Resultado leído de Drive"] = False
    print(f"FAIL: no se pudo verificar el estado final en Drive: {error}")

if not all(checks.values()):
    payload["final_status"] = "FAIL"
print(f"JSON: {result_path}")

JSON: /content/drive/MyDrive/SLM/results/integration_test/colab_integration_20261025_131104.json


In [24]:
print("COLAB INTEGRATION TEST\n")
for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

final_status = "PASS" if all(checks.values()) else "FAIL"
print(f"\nFINAL: {final_status}")
if final_status != "PASS":
    failed_checks = [name for name, passed in checks.items() if not passed]
    raise RuntimeError(
        "La prueba de integración falló: " + ", ".join(failed_checks)
    )

COLAB INTEGRATION TEST

[PASS] GPU disponible
[PASS] Repositorio accesible
[PASS] Paquete slm importable
[PASS] Código de src/slm ejecutado
[PASS] Operación CUDA ejecutada
[PASS] Google Drive montado
[PASS] Resultado escrito en Drive
[PASS] Resultado leído de Drive

FINAL: PASS
